# LeetCode 1313 — Decompress Run-Length Encoded List bằng Python

**Link đề:** https://leetcode.com/problems/decompress-run-length-encoded-list/
**Độ khó:** Easy · **Chủ đề:** Array (duyệt theo cặp, nén run-length)

```python
class Solution:
    def decompressRLElist(self, nums: list[int]) -> list[int]:
        ...
```

## Mục tiêu học
1. Hiểu **Run-Length Encoding (RLE)** — cách nén "lặp lại bao nhiêu lần, giá trị gì".
2. Biết duyệt list **theo từng cặp** bằng `range(0, n, 2)`, slicing bước nhảy `nums[::2]`, `zip`.
3. Nắm các cách tạo phần tử lặp: vòng lặp `append`, `[val] * freq`, `extend`, list comprehension lồng.
4. Trace, chứng minh tính đúng, Time/Space (theo **độ dài output**, không chỉ theo input!).
5. Luyện biến thể: nén ngược (encode), giải nén chuỗi `"3a2b"`, truy vấn phần tử thứ `k` mà không giải nén.

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

Cho list `nums` biểu diễn một list đã được nén bằng **run-length encoding**.

Xét từng cặp liền kề `[freq, val] = [nums[2*i], nums[2*i+1]]` (với `i >= 0`). Mỗi cặp sinh ra một list con gồm `freq` phần tử có giá trị `val`. Nối tất cả list con từ trái sang phải để được list giải nén.

| Input `nums` | Output | Giải thích |
|---|---|---|
| `[1,2,3,4]` | `[2,4,4,4]` | cặp `[1,2]` → `[2]`; cặp `[3,4]` → `[4,4,4]` |
| `[1,1,2,3]` | `[1,3,3]` | `[1,1]` → `[1]`; `[2,3]` → `[3,3]` |

**Constraints:** `2 <= nums.length <= 100`, `nums.length % 2 == 0`, `1 <= nums[i] <= 100`.

Câu hỏi trước khi code:
- Phần tử ở vị trí **chẵn** là gì, ở vị trí **lẻ** là gì?
- Output dài tối đa bao nhiêu? (gợi ý: 50 cặp × 100 lần)

## 2. Nền tảng: Run-Length Encoding là gì?

RLE là cách nén dữ liệu có **nhiều phần tử lặp liên tiếp**: thay vì ghi `4,4,4,4,4`, ta ghi `(5 lần, giá trị 4)`.

Ví dụ list `[7, 7, 7, 2, 9, 9]` → RLE: `[3, 7, 1, 2, 2, 9]`.

Bài này là chiều **ngược lại**: từ dạng nén, khôi phục list gốc.

In [1]:
original = [7, 7, 7, 2, 9, 9]
encoded = [3, 7, 1, 2, 2, 9]
print("gốc   :", original, "-", len(original), "phần tử")
print("nén   :", encoded, "-", len(encoded), "phần tử")
print("cặp (freq, val):", [(encoded[i], encoded[i + 1]) for i in range(0, len(encoded), 2)])

gốc   : [7, 7, 7, 2, 9, 9] - 6 phần tử
nén   : [3, 7, 1, 2, 2, 9] - 6 phần tử
cặp (freq, val): [(3, 7), (1, 2), (2, 9)]


### 2.1. `range(0, n, 2)` — nhảy 2 bước để đọc theo cặp

`range(0, 6, 2)` → `0, 2, 4`. Tại mỗi `i` chẵn: `nums[i]` là `freq`, `nums[i + 1]` là `val`.

Vì `len(nums)` luôn chẵn, `i + 1` không bao giờ vượt biên.

In [2]:
nums = [1, 2, 3, 4]
print(list(range(0, len(nums), 2)))
for i in range(0, len(nums), 2):
    print(f"i={i}: freq = nums[{i}] = {nums[i]}, val = nums[{i + 1}] = {nums[i + 1]}")

[0, 2]
i=0: freq = nums[0] = 1, val = nums[1] = 2
i=2: freq = nums[2] = 3, val = nums[3] = 4


### 2.2. Nhân list: `[val] * freq`

`[4] * 3` → `[4, 4, 4]`. Phép `*` trên list tạo list mới lặp lại nội dung `freq` lần. `[4] * 0` → `[]`.

In [3]:
print([4] * 3)
print([2] * 1)
print(['x', 'y'] * 2)
print([4] * 0)

[4, 4, 4]
[2]
['x', 'y', 'x', 'y']
[]


### 2.3. `append` vs `extend` vs `+=`

- `a.append(x)` thêm **một** phần tử `x` (kể cả nếu `x` là list → thành list lồng!).
- `a.extend(lst)` thêm **từng phần tử** của `lst` vào cuối `a`.
- `a += lst` tương đương `a.extend(lst)`.

In [4]:
a = [2]
a.append([4, 4, 4])
print("append list:", a, "<- list lồng, sai ý!")
a = [2]
a.extend([4, 4, 4])
print("extend     :", a)
a = [2]
a += [4, 4, 4]
print("+=         :", a)

append list: [2, [4, 4, 4]] <- list lồng, sai ý!
extend     : [2, 4, 4, 4]
+=         : [2, 4, 4, 4]


### 2.4. Slicing với bước nhảy: `nums[::2]` và `nums[1::2]`

- `nums[::2]` lấy các phần tử ở vị trí chẵn `0, 2, 4, ...` → toàn bộ `freq`.
- `nums[1::2]` lấy vị trí lẻ `1, 3, 5, ...` → toàn bộ `val`.

In [5]:
nums = [3, 7, 1, 2, 2, 9]
print("freqs:", nums[::2])
print("vals :", nums[1::2])

freqs: [3, 1, 2]
vals : [7, 2, 9]


### 2.5. `zip(a, b)` — ghép hai list thành các cặp

`zip([3,1,2], [7,2,9])` cho các cặp `(3,7), (1,2), (2,9)`. Kết hợp với 2.4: `zip(nums[::2], nums[1::2])` duyệt đúng từng cặp `(freq, val)`.

In [6]:
nums = [3, 7, 1, 2, 2, 9]
for freq, val in zip(nums[::2], nums[1::2]):
    print("freq =", freq, "val =", val)
print(list(zip([3, 1, 2], [7, 2, 9])))

freq = 3 val = 7
freq = 1 val = 2
freq = 2 val = 9
[(3, 7), (1, 2), (2, 9)]


Ở `for freq, val in ...` ta dùng **unpacking**: mỗi phần tử là tuple 2 giá trị, Python tách vào 2 biến.

## 3. Cách trực giác / baseline — hai vòng lặp lồng

Người mới nghĩ thẳng theo đề: "với mỗi cặp, lặp `freq` lần, mỗi lần `append(val)`".

In [7]:
def decompress_nested(nums: list[int]) -> list[int]:
    result = []
    for i in range(0, len(nums), 2):
        freq, val = nums[i], nums[i + 1]
        for _ in range(freq):          # '_' = biến không dùng đến
            result.append(val)
    return result

print(decompress_nested([1, 2, 3, 4]))
print(decompress_nested([1, 1, 2, 3]))

[2, 4, 4, 4]
[1, 3, 3]


Tên biến `_` là quy ước "tôi không cần giá trị này, chỉ cần lặp đủ số lần".

### 3.1. Baseline có chậm không?

Hai vòng lồng nhưng **không phải** `O(n²)`! Tổng số lần `append` đúng bằng **độ dài output** `L = tổng các freq`. Mọi lời giải đều phải tạo ra `L` phần tử, nên baseline đã tối ưu về Big-O.

Điểm có thể cải thiện chỉ là **hằng số và độ gọn**: `append` từng phần tử trong vòng lặp Python chậm hơn `extend([val] * freq)` (thực hiện bên trong C).

In [8]:
nums = [1, 2, 3, 4, 100, 7]
L = sum(nums[::2])
print("độ dài output L =", L, "| len(output) =", len(decompress_nested(nums)))

độ dài output L = 104 | len(output) = 104


### 3.2. Đo thử: `append` từng phần tử vs `extend` cả khối

Module chuẩn `timeit` chạy một hàm nhiều lần và đo tổng thời gian. Ta tạo input "xấu nhất" theo constraints: 50 cặp, mỗi `freq = 100` → output 5000 phần tử.

Con số cụ thể phụ thuộc máy, nhưng thường `extend` nhanh hơn vài lần vì vòng lặp chạy trong C thay vì trong Python. Big-O **giống nhau**; khác nhau chỉ ở hằng số.

In [9]:
import timeit

worst = [100, 7] * 50

def by_append(nums):
    result = []
    for i in range(0, len(nums), 2):
        for _ in range(nums[i]):
            result.append(nums[i + 1])
    return result

def by_extend(nums):
    result = []
    for i in range(0, len(nums), 2):
        result.extend([nums[i + 1]] * nums[i])
    return result

assert by_append(worst) == by_extend(worst)
t1 = timeit.timeit(lambda: by_append(worst), number=200)
t2 = timeit.timeit(lambda: by_extend(worst), number=200)
print("append từng phần tử nhanh hơn?", t1 < t2, "| extend nhanh hơn?", t2 < t1)
print("cả hai cho output dài", len(by_extend(worst)))

append từng phần tử nhanh hơn? False | extend nhanh hơn? True
cả hai cho output dài 5000


`lambda: ...` là **hàm vô danh** một dòng — ở đây dùng để "gói" lời gọi hàm cho `timeit` gọi lại nhiều lần.

## 4. Insight — mỗi cặp là một "khối" `[val] * freq`

Thay vì lặp từng phần tử, ta tạo **cả khối** trong một lệnh: `[val] * freq`, rồi `extend` vào kết quả. Nối các khối theo thứ tự chính là định nghĩa của đề.

In [10]:
nums = [1, 2, 3, 4]
blocks = [[nums[i + 1]] * nums[i] for i in range(0, len(nums), 2)]
print("các khối:", blocks)
result = []
for blk in blocks:
    result.extend(blk)
print("nối lại :", result)

các khối: [[2], [4, 4, 4]]
nối lại : [2, 4, 4, 4]


## 5. Lời giải chính — `extend` từng khối

**Đoạn 1:** `result = []`.

**Đoạn 2:** duyệt `i` trong `range(0, len(nums), 2)`, lấy `freq = nums[i]`, `val = nums[i + 1]`.

**Đoạn 3:** `result.extend([val] * freq)`.

**Đoạn 4:** `return result`.

In [11]:
def decompress_extend(nums: list[int]) -> list[int]:
    result = []
    for i in range(0, len(nums), 2):
        freq, val = nums[i], nums[i + 1]
        result.extend([val] * freq)
    return result

for nums in [[1, 2, 3, 4], [1, 1, 2, 3], [5, 9], [1, 1, 1, 1]]:
    print(nums, "->", decompress_extend(nums))

[1, 2, 3, 4] -> [2, 4, 4, 4]
[1, 1, 2, 3] -> [1, 3, 3]
[5, 9] -> [9, 9, 9, 9, 9]
[1, 1, 1, 1] -> [1, 1]


## 6. TRACE từng bước

Trace `nums = [1, 2, 3, 4]`:

| i | freq = nums[i] | val = nums[i+1] | khối `[val]*freq` | result sau extend |
|---|---|---|---|---|
| 0 | 1 | 2 | `[2]` | `[2]` |
| 2 | 3 | 4 | `[4,4,4]` | `[2,4,4,4]` |

`i = 4 = len(nums)` → `range` kết thúc.

In [12]:
def trace_decompress(nums: list[int]) -> list[int]:
    print("nums =", nums)
    result = []
    for i in range(0, len(nums), 2):
        freq, val = nums[i], nums[i + 1]
        block = [val] * freq
        result.extend(block)
        print(f"  i={i}: freq={freq}, val={val}, khối={block}, result={result}")
    print("  kết quả:", result)
    return result

trace_decompress([1, 2, 3, 4])
print()
trace_decompress([2, 5, 1, 9, 3, 5])

nums = [1, 2, 3, 4]
  i=0: freq=1, val=2, khối=[2], result=[2]
  i=2: freq=3, val=4, khối=[4, 4, 4], result=[2, 4, 4, 4]
  kết quả: [2, 4, 4, 4]

nums = [2, 5, 1, 9, 3, 5]
  i=0: freq=2, val=5, khối=[5, 5], result=[5, 5]
  i=2: freq=1, val=9, khối=[9], result=[5, 5, 9]
  i=4: freq=3, val=5, khối=[5, 5, 5], result=[5, 5, 9, 5, 5, 5]
  kết quả: [5, 5, 9, 5, 5, 5]


[5, 5, 9, 5, 5, 5]

Ví dụ thứ hai cho thấy: hai khối cùng giá trị `5` **không bị gộp** — thứ tự được giữ nguyên `[5,5,9,5,5,5]`.

## 7. Code nộp LeetCode

In [13]:
class Solution:
    def decompressRLElist(self, nums: list[int]) -> list[int]:
        result = []
        for i in range(0, len(nums), 2):
            result.extend([nums[i + 1]] * nums[i])
        return result

print(Solution().decompressRLElist([1, 2, 3, 4]))
print(Solution().decompressRLElist([1, 1, 2, 3]))

[2, 4, 4, 4]
[1, 3, 3]


## 8. Các lời giải khác

### Cách 3 — `zip` + slicing bước nhảy

Đọc cặp `(freq, val)` bằng `zip(nums[::2], nums[1::2])` — không phải đụng chỉ số `i`, khó sai biên.

In [14]:
def decompress_zip(nums: list[int]) -> list[int]:
    result = []
    for freq, val in zip(nums[::2], nums[1::2]):
        result += [val] * freq
    return result

print(decompress_zip([1, 2, 3, 4]), decompress_zip([1, 1, 2, 3]))

[2, 4, 4, 4] [1, 3, 3]


### Cách 4 — list comprehension lồng (một dòng)

```python
[val for i in range(0, len(nums), 2) for val in [nums[i + 1]] * nums[i]]
```

Đọc từ trái qua: "với mỗi `i`, với mỗi `val` trong khối, lấy `val`". Thứ tự `for` trong comprehension giống thứ tự viết vòng lồng thông thường.

In [15]:
def decompress_comprehension(nums: list[int]) -> list[int]:
    return [val for i in range(0, len(nums), 2) for val in [nums[i + 1]] * nums[i]]

def decompress_comprehension2(nums: list[int]) -> list[int]:
    return [val for freq, val in zip(nums[::2], nums[1::2]) for _ in range(freq)]

print(decompress_comprehension([1, 2, 3, 4]))
print(decompress_comprehension2([1, 1, 2, 3]))

[2, 4, 4, 4]
[1, 3, 3]


### Cách 5 — `itertools` (thư viện chuẩn)

- `itertools.repeat(val, freq)` sinh `val` lặp `freq` lần (không tạo list ngay).
- `itertools.chain.from_iterable(...)` nối nhiều dãy thành một dãy.

In [16]:
from itertools import chain, repeat

def decompress_itertools(nums: list[int]) -> list[int]:
    return list(chain.from_iterable(repeat(v, f) for f, v in zip(nums[::2], nums[1::2])))

print(list(repeat(4, 3)))
print(decompress_itertools([1, 2, 3, 4]), decompress_itertools([1, 1, 2, 3]))

[4, 4, 4]
[2, 4, 4, 4] [1, 3, 3]


### Code nộp LeetCode cho các cách khác

In [17]:
class SolutionZip:
    def decompressRLElist(self, nums: list[int]) -> list[int]:
        result = []
        for freq, val in zip(nums[::2], nums[1::2]):
            result += [val] * freq
        return result

class SolutionOneLine:
    def decompressRLElist(self, nums: list[int]) -> list[int]:
        return [v for f, v in zip(nums[::2], nums[1::2]) for _ in range(f)]

class SolutionNested:
    def decompressRLElist(self, nums: list[int]) -> list[int]:
        result = []
        for i in range(0, len(nums), 2):
            for _ in range(nums[i]):
                result.append(nums[i + 1])
        return result

for cls in [SolutionZip, SolutionOneLine, SolutionNested]:
    print(cls.__name__, cls().decompressRLElist([1, 2, 3, 4]))

SolutionZip [2, 4, 4, 4]
SolutionOneLine [2, 4, 4, 4]
SolutionNested [2, 4, 4, 4]


### Bảng so sánh

| Cách | Ý tưởng | Time | Space phụ | Ghi chú |
|---|---|---|---|---|
| 1. Lồng `append` | lặp freq lần | `O(n + L)` | `O(1)` | dễ nhất, chậm hằng số |
| 2. `extend([v]*f)` | tạo khối rồi nối | `O(n + L)` | `O(max f)` khối tạm | lời giải chính |
| 3. `zip` + slicing | đọc cặp không cần chỉ số | `O(n + L)` | `O(n)` cho 2 slice | gọn, khó sai biên |
| 4. Comprehension | một dòng | `O(n + L)` | `O(1)` | ngắn, cần đọc quen |
| 5. `itertools` | `chain` + `repeat` | `O(n + L)` | `O(1)` | phong cách "lazy" |

`n = len(nums)`, `L = tổng các freq` = độ dài output. Không tính output vào space phụ.

## 9. Vì sao lời giải đúng?

**Invariant:** sau khi xử lý cặp thứ `k` (tức `i = 2k`), `result` bằng **nối các khối của cặp 0..k theo đúng thứ tự**.

- Ban đầu `result = []` — nối 0 khối.
- Mỗi bước `extend` khối `[val]*freq` vào **cuối**, nên thứ tự từ trái sang phải được giữ.
- `range(0, len(nums), 2)` đi qua mọi chỉ số chẵn, không bỏ cặp nào; vì `len(nums)` chẵn nên `i + 1` luôn hợp lệ.

Kết thúc: `result` là nối mọi khối — đúng định nghĩa đề. ∎

### 9.1. Độ phức tạp

- **Time:** `O(n + L)` — duyệt `n/2` cặp, và tạo tổng cộng `L` phần tử. Với constraints: `n <= 100`, `L <= 50 × 100 = 5000`.
- **Space:** output `O(L)` (bắt buộc). Phụ: `O(f)` cho khối tạm `[val]*freq` (cách 2), `O(1)` với vòng lồng.

Lưu ý quan trọng: độ phức tạp phụ thuộc **kích thước output**, không chỉ input — đặc trưng của bài "giải nén".

## 10. Bộ test

Test mẫu đề, biên (1 cặp, freq = 1, freq = 100), giá trị trùng giữa các khối, và **đối chiếu mọi cách** trên input ngẫu nhiên. Thêm kiểm tra **khứ hồi** (round trip): nén rồi giải nén phải ra list gốc.

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<22} PASS")

sol = Solution()
check("ví dụ 1", sol.decompressRLElist([1, 2, 3, 4]), [2, 4, 4, 4])
check("ví dụ 2", sol.decompressRLElist([1, 1, 2, 3]), [1, 3, 3])
check("1 cặp freq=1", sol.decompressRLElist([1, 5]), [5])
check("1 cặp freq=100", sol.decompressRLElist([100, 7]), [7] * 100)
check("khối trùng giá trị", sol.decompressRLElist([2, 5, 1, 9, 3, 5]), [5, 5, 9, 5, 5, 5])
check("độ dài tối đa", len(sol.decompressRLElist([100, 1] * 50)), 5000)

ví dụ 1                PASS
ví dụ 2                PASS
1 cặp freq=1           PASS
1 cặp freq=100         PASS
khối trùng giá trị     PASS
độ dài tối đa          PASS


In [19]:
import random

def rle_encode(arr: list[int]) -> list[int]:
    # nén list thành dạng [freq, val, freq, val, ...] (gộp các phần tử lặp liên tiếp)
    out = []
    for x in arr:
        if out and out[-1] == x:
            out[-2] += 1
        else:
            out += [1, x]
    return out

print("rle_encode([7,7,7,2,9,9]) =", rle_encode([7, 7, 7, 2, 9, 9]))

rng = random.Random(1313)
methods = [decompress_nested, decompress_extend, decompress_zip, decompress_comprehension,
           decompress_comprehension2, decompress_itertools, Solution().decompressRLElist]
for _ in range(300):
    nums = [rng.randint(1, 100) for _ in range(2 * rng.randint(1, 50))]
    expected = decompress_nested(nums)
    for f in methods:
        assert f(nums) == expected
    arr = [rng.randint(1, 3) for _ in range(rng.randint(1, 30))]
    assert decompress_extend(rle_encode(arr)) == arr          # khứ hồi
print("300 input ngẫu nhiên: mọi cách khớp nhau + khứ hồi đúng")

rle_encode([7,7,7,2,9,9]) = [3, 7, 1, 2, 2, 9]
300 input ngẫu nhiên: mọi cách khớp nhau + khứ hồi đúng


## 11. Lỗi người mới hay gặp

### Lỗi 1: `append` một list → list lồng

In [20]:
def wrong_append_block(nums):
    result = []
    for i in range(0, len(nums), 2):
        result.append([nums[i + 1]] * nums[i])   # sai: thêm cả list như 1 phần tử
    return result

print("sai :", wrong_append_block([1, 2, 3, 4]))
print("đúng:", decompress_extend([1, 2, 3, 4]))

sai : [[2], [4, 4, 4]]
đúng: [2, 4, 4, 4]


**Sửa:** dùng `extend` hoặc `+=`.

### Lỗi 2: Đảo vai trò `freq` và `val`

Đề ghi `[freq, val]` — **số lần đứng trước**. Đảo lại sẽ sinh `[1, 1, 3, 3, 3, 3]` cho ví dụ 1 (lặp `1` hai lần, `3` bốn lần).

### Lỗi 3: Duyệt `range(len(nums))` (bước 1)

Khi đó `i = 1` coi `nums[1]` là `freq`, xử lý lệch cặp, và `i = len-1` đọc `nums[len]` → `IndexError`.

In [21]:
def wrong_swap(nums):
    result = []
    for i in range(0, len(nums), 2):
        result.extend([nums[i]] * nums[i + 1])
    return result

def wrong_step_one(nums):
    result = []
    for i in range(len(nums)):
        result.extend([nums[i + 1]] * nums[i])
    return result

print("đảo vai trò:", wrong_swap([1, 2, 3, 4]))
try:
    wrong_step_one([1, 2, 3, 4])
except IndexError as e:
    print("bước 1 -> IndexError:", e)

đảo vai trò: [1, 1, 3, 3, 3, 3]
bước 1 -> IndexError: list index out of range


### Lỗi 4: Nhầm `[val] * freq` với `val * freq`

`4 * 3` là số `12`, không phải list `[4, 4, 4]`. Nhớ bọc `val` trong `[...]`.

In [22]:
print("val * freq   =", 4 * 3)
print("[val] * freq =", [4] * 3)

val * freq   = 12
[val] * freq = [4, 4, 4]


## 12. Mẹo nhận dạng pattern

- Mảng mà **vị trí chẵn và lẻ có ý nghĩa khác nhau** ⇒ duyệt `range(0, n, 2)` hoặc `zip(a[::2], a[1::2])`.
- "Lặp giá trị `k` lần" ⇒ `[x] * k` hoặc `itertools.repeat`.
- Bài "giải nén / mở rộng" ⇒ phân tích độ phức tạp theo **kích thước output**.
- Khi có cặp hàm encode/decode ⇒ kiểm thử **khứ hồi** rất mạnh và dễ viết.

## 13. Biến thể

1. **Nén (encode)** ngược lại — đã viết `rle_encode` ở trên.
2. **Giải nén chuỗi** dạng `"3a2b1c"` → `"aaabbc"` (LeetCode 443 String Compression là chiều nén).
3. **Truy vấn phần tử thứ `k`** mà không giải nén cả list (LeetCode 900 RLE Iterator, 880 Decoded String at Index): đi qua từng cặp, trừ dần `k`.

In [23]:
def kth_without_decompress(nums: list[int], k: int) -> int:
    # k tính từ 0; không tạo list giải nén
    for freq, val in zip(nums[::2], nums[1::2]):
        if k < freq:
            return val
        k -= freq
    raise IndexError("k vượt độ dài")

nums = [1, 2, 3, 4]
print([kth_without_decompress(nums, k) for k in range(4)], "==", decompress_extend(nums))

[2, 4, 4, 4] == [2, 4, 4, 4]


## 14. Bài tập tự luyện (TODO — tăng dần)

**Bài 1 (tính tay):** Giải nén `[2, 8, 1, 0, 3, 8]`. Output có bao nhiêu phần tử?

**Bài 2 (đoán output):** `[0, 1] * 2` và `[[0] * 2] * 2` in ra gì?

**Bài 3 (tự viết):** `exercise_decompress(nums)` dùng `zip` và `extend`.

**Bài 4 (đảo chiều):** `exercise_encode(arr)` — nén list thành dạng RLE của đề (gộp các phần tử lặp **liên tiếp**).

**Bài 5 (biến thể chuỗi):** `exercise_decode_string("3a2b1c")` → `"aaabbc"` (số lần có thể nhiều chữ số, ví dụ `"12x"`).

**Bài 6 (mini challenge):** `exercise_length_and_kth(nums, k)` trả về `(độ dài output, phần tử thứ k)` mà **không** tạo list giải nén.

In [24]:
def exercise_decompress(nums: list[int]) -> list[int]:
    # TODO
    pass

def exercise_encode(arr: list[int]) -> list[int]:
    # TODO: gộp các phần tử lặp liên tiếp thành [freq, val]
    pass

# assert exercise_decompress([1, 2, 3, 4]) == [2, 4, 4, 4]
# assert exercise_encode([2, 4, 4, 4]) == [1, 2, 3, 4]

In [25]:
def exercise_decode_string(s: str) -> str:
    # TODO: đọc số (có thể nhiều chữ số) rồi một chữ cái
    pass

def exercise_length_and_kth(nums: list[int], k: int) -> tuple[int, int]:
    # TODO: không tạo list giải nén
    pass

# assert exercise_decode_string("3a2b1c") == "aaabbc"
# assert exercise_decode_string("12x") == "x" * 12
# assert exercise_length_and_kth([1, 2, 3, 4], 2) == (4, 4)

## 15. Gợi ý 3 mức

**Bài 1** — (1) Cặp: `(2,8), (1,0), (3,8)`. (2) Khối `8` không gộp với khối `8` sau. (3) Độ dài = `2+1+3`.

**Bài 2** — (1) Nhân list lặp lại *nội dung*. (2) `[0]*2` là `[0,0]`. (3) Lồng thêm một lớp.

**Bài 4** — (1) Duyệt từng `x`. (2) Nếu `x` bằng `val` của cặp cuối thì tăng `freq`. (3) `out[-1] == x` → `out[-2] += 1`, ngược lại `out += [1, x]`.

**Bài 5** — (1) Gom chữ số vào biến `count`. (2) `count = count * 10 + int(ch)`. (3) Gặp chữ cái: thêm `ch * count`, reset `count = 0`.

**Bài 6** — (1) Độ dài = `sum(nums[::2])`. (2) Đi qua từng cặp, nếu `k < freq` thì trả `val`. (3) Ngược lại `k -= freq`.

## 16. Đáp án tham khảo

**Bài 1:** `[8, 8, 0, 8, 8, 8]`, 6 phần tử.

**Bài 2:** `[0, 1, 0, 1]` và `[[0, 0], [0, 0]]` (lưu ý: hai list con trong `[[0]*2]*2` là **cùng một đối tượng** — sửa một cái sẽ thấy cả hai đổi).

In [26]:
def answer_decompress(nums):
    result = []
    for f, v in zip(nums[::2], nums[1::2]):
        result.extend([v] * f)
    return result

def answer_encode(arr):
    out = []
    for x in arr:
        if out and out[-1] == x:
            out[-2] += 1
        else:
            out += [1, x]
    return out

def answer_decode_string(s):
    out, count = [], 0
    for ch in s:
        if ch.isdigit():
            count = count * 10 + int(ch)
        else:
            out.append(ch * count)
            count = 0
    return ''.join(out)

def answer_length_and_kth(nums, k):
    total = sum(nums[::2])
    for f, v in zip(nums[::2], nums[1::2]):
        if k < f:
            return total, v
        k -= f
    raise IndexError

assert answer_decompress([2, 8, 1, 0, 3, 8]) == [8, 8, 0, 8, 8, 8]
assert [0, 1] * 2 == [0, 1, 0, 1] and [[0] * 2] * 2 == [[0, 0], [0, 0]]
assert answer_decompress([1, 2, 3, 4]) == [2, 4, 4, 4]
assert answer_encode([2, 4, 4, 4]) == [1, 2, 3, 4]
assert answer_decompress(answer_encode([5, 5, 1, 5])) == [5, 5, 1, 5]
assert answer_decode_string("3a2b1c") == "aaabbc"
assert answer_decode_string("12x") == "x" * 12
assert answer_length_and_kth([1, 2, 3, 4], 2) == (4, 4)
for k in range(9):
    assert answer_length_and_kth([2, 8, 1, 0, 3, 8, 3, 1], k)[1] == answer_decompress([2, 8, 1, 0, 3, 8, 3, 1])[k]
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


## 17. Template nhớ nhanh + checklist submit

```python
class Solution:
    def decompressRLElist(self, nums: list[int]) -> list[int]:
        result = []
        for i in range(0, len(nums), 2):        # bước 2: đọc theo cặp
            freq, val = nums[i], nums[i + 1]    # freq đứng TRƯỚC
            result.extend([val] * freq)         # extend, không append
        return result
```

**Checklist:**
- [ ] Duyệt bước 2 (`range(0, n, 2)`), không phải bước 1.
- [ ] `nums[i]` là `freq`, `nums[i+1]` là `val`.
- [ ] Dùng `extend` / `+=`, không `append` cả khối.
- [ ] `[val] * freq` có ngoặc vuông.
- [ ] Hiểu rằng Time phụ thuộc độ dài output `L`.